Copyright (c) MONAI Consortium  
Licensed under the Apache License, Version 2.0 (the "License");  
you may not use this file except in compliance with the License.  
You may obtain a copy of the License at  
&nbsp;&nbsp;&nbsp;&nbsp;http://www.apache.org/licenses/LICENSE-2.0  
Unless required by applicable law or agreed to in writing, software  
distributed under the License is distributed on an "AS IS" BASIS,  
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.  
See the License for the specific language governing permissions and  
limitations under the License.

# MONAI 201 tutorial

In this tutorial we'll revisit the [MONAI 101 notebook](https://github.com/Project-MONAI/tutorials/blob/main/2d_classification/monai_101.ipynb) and add more features representing best practice concepts. This will include evaluation and tensorboard handler techniques.

These steps will be included in this tutorial, and each of them will take only a few lines of code:
- Dataset download and Data pre-processing
- Define a DenseNet-121 and run training
- Run inference using SupervisedEvaluator

This tutorial will use about 7GB of GPU memory and 10 minutes to run.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Project-MONAI/tutorials/blob/main/2d_classification/monai_201.ipynb)

## Setup environment

In [1]:
!python -c "import monai" || pip install -q "monai-weekly[ignite, tqdm, tensorboard]"

2024-11-08 13:19:00.564540: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-08 13:19:00.575163: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731068340.588624  187607 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731068340.592625  187607 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-08 13:19:00.605894: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

## Setup imports

In [2]:
import logging
import numpy as np
import os
from pathlib import Path
import sys
import tempfile
import torch
import ignite

from monai.apps import MedNISTDataset
from monai.config import print_config
from monai.data import DataLoader
from monai.engines import SupervisedTrainer, SupervisedEvaluator
from monai.handlers import (
    StatsHandler,
    TensorBoardStatsHandler,
    ValidationHandler,
    CheckpointSaver,
    CheckpointLoader,
    ClassificationSaver,
)
from monai.handlers.utils import from_engine
from monai.inferers import SimpleInferer
from monai.networks.nets import densenet121
from monai.transforms import LoadImageD, EnsureChannelFirstD, ScaleIntensityD, Compose, AsDiscreted

print_config()

/home/arnaud/miniconda3/envs/mri/lib/python3.10/site-packages/ignite/handlers/checkpoint.py:17: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer
2024-11-08 13:19:36.990988: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-08 13:19:37.001764: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731068377.015646  187575 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has al

MONAI version: 1.4.0
Numpy version: 1.26.4
Pytorch version: 2.5.1+cu124
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 46a5272196a6c2590ca2589029eed8e4d56ff008
MONAI __file__: /home/<username>/miniconda3/envs/mri/lib/python3.10/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: 0.4.11
ITK version: 5.4.0
Nibabel version: 5.0.0
scikit-image version: 0.24.0
scipy version: 1.14.1
Pillow version: 11.0.0
Tensorboard version: 2.18.0
gdown version: 5.2.0
TorchVision version: 0.20.1+cu124
tqdm version: 4.66.6
lmdb version: 1.5.1
psutil version: 6.1.0
pandas version: 2.2.3
einops version: 0.8.0
transformers version: 4.40.2
mlflow version: 2.17.2
pynrrd version: 1.0.0
clearml version: 1.16.5

For details about installing the optional dependencies, please visit:
    https://docs.monai.io/en/latest/installation.html#installing-the-recommended-dependencies



## Setup data directory

You can specify a directory with the `MONAI_DATA_DIRECTORY` environment variable.  
This allows you to save results and reuse downloads.  
If not specified a temporary directory will be used.

In [3]:
directory = os.environ.get("MONAI_DATA_DIRECTORY")
if directory is not None:
    os.makedirs(directory, exist_ok=True)
root_dir = tempfile.mkdtemp() if directory is None else directory
print(root_dir)

/tmp/tmprebutymo


## Use MONAI transforms to preprocess data

We'll first prepare the data very much like in the previous tutorial with the same transforms and dataset:

In [4]:
transform = Compose(
    [
        LoadImageD(keys="image", image_only=True),
        EnsureChannelFirstD(keys="image"),
        ScaleIntensityD(keys="image"),
    ]
)

# If you use the MedNIST dataset, please acknowledge the source.
dataset = MedNISTDataset(root_dir=root_dir, transform=transform, section="training", download=True)
valdata = MedNISTDataset(root_dir=root_dir, transform=transform, section="validation", download=False)

MedNIST.tar.gz: 59.0MB [00:01, 33.0MB/s]                              

2024-11-08 13:19:53,887 - INFO - Downloaded: /tmp/tmprebutymo/MedNIST.tar.gz


2024-11-08 13:19:53,989 - INFO - Verified 'MedNIST.tar.gz', md5: 0bc7306e7427e00ad1c5526a6677552d.
2024-11-08 13:19:53,990 - INFO - Writing into directory: /tmp/tmprebutymo.


Loading dataset: 100%|██████████| 5895/5895 [00:04<00:00, 1208.98it/s]


## Define a network and a supervised trainer

For training we have the same elements again and will slightly change the `SupervisedTrainer` by expanding its train_handlers. This upgrade will be beneficial for efficient utilization of TensorBoard.
Furthermore, we introduce a `SupervisedEvaluator` object that will efficiently track model progress. Accompanied by `TensorBoardStatsHandler`, it will log statistics for TensorBoard, ensuring precise tracking and management.

In [5]:
max_epochs = 5
save_interval = 2
out_dir = "./eval"
model = densenet121(spatial_dims=2, in_channels=1, out_channels=6).to("cuda:0")

logging.basicConfig(stream=sys.stdout, level=logging.INFO)

evaluator = SupervisedEvaluator(
    device=torch.device("cuda:0"),
    val_data_loader=DataLoader(valdata, batch_size=512, shuffle=False, num_workers=4),
    network=model,
    inferer=SimpleInferer(),
    key_val_metric={"val_acc": ignite.metrics.Accuracy(from_engine(["pred", "label"]))},
    val_handlers=[StatsHandler(iteration_log=False), TensorBoardStatsHandler(iteration_log=False)],
)

trainer = SupervisedTrainer(
    device=torch.device("cuda:0"),
    max_epochs=max_epochs,
    train_data_loader=DataLoader(dataset, batch_size=512, shuffle=True, num_workers=4),
    network=model,
    optimizer=torch.optim.Adam(model.parameters(), lr=1e-5),
    loss_function=torch.nn.CrossEntropyLoss(),
    inferer=SimpleInferer(),
    train_handlers=[
        ValidationHandler(validator=evaluator, epoch_level=True, interval=1),
        CheckpointSaver(
            save_dir=out_dir,
            save_dict={"model": model},
            save_interval=save_interval,
            save_final=True,
            final_filename="checkpoint.pt",
        ),
        StatsHandler(),
        TensorBoardStatsHandler(tag_name="train_loss", output_transform=from_engine(["loss"], first=True)),
    ],
)

## Run the training

In [6]:
trainer.run()

INFO:ignite.engine.engine.SupervisedTrainer:Engine run resuming from iteration 0, epoch 0 until 5 epochs
2024-11-08 13:27:09,118 - INFO - Epoch: 1/5, Iter: 1/93 -- label: 0.0000 loss: 1.8519 
2024-11-08 13:27:09,302 - INFO - Epoch: 1/5, Iter: 2/93 -- label: 4.0000 loss: 1.7928 


ignoring non-scalar output in StatsHandler, make sure `output_transform(engine.state.output)` returns a scalar or dictionary of key and scalar pairs to avoid this warning. image:<class 'monai.data.meta_tensor.MetaTensor'>
ignoring non-scalar output in StatsHandler, make sure `output_transform(engine.state.output)` returns a scalar or dictionary of key and scalar pairs to avoid this warning. pred:<class 'monai.data.meta_tensor.MetaTensor'>


2024-11-08 13:27:09,487 - INFO - Epoch: 1/5, Iter: 3/93 -- label: 0.0000 loss: 1.7846 
2024-11-08 13:27:09,659 - INFO - Epoch: 1/5, Iter: 4/93 -- label: 2.0000 loss: 1.7442 
2024-11-08 13:27:09,859 - INFO - Epoch: 1/5, Iter: 5/93 -- label: 1.0000 loss: 1.7173 
2024-11-08 13:27:10,040 - INFO - Epoch: 1/5, Iter: 6/93 -- label: 4.0000 loss: 1.6661 
2024-11-08 13:27:10,239 - INFO - Epoch: 1/5, Iter: 7/93 -- label: 5.0000 loss: 1.6705 
2024-11-08 13:27:10,415 - INFO - Epoch: 1/5, Iter: 8/93 -- label: 5.0000 loss: 1.6466 
2024-11-08 13:27:10,585 - INFO - Epoch: 1/5, Iter: 9/93 -- label: 0.0000 loss: 1.6055 
2024-11-08 13:27:10,756 - INFO - Epoch: 1/5, Iter: 10/93 -- label: 0.0000 loss: 1.5618 
2024-11-08 13:27:10,935 - INFO - Epoch: 1/5, Iter: 11/93 -- label: 1.0000 loss: 1.5386 
2024-11-08 13:27:11,115 - INFO - Epoch: 1/5, Iter: 12/93 -- label: 2.0000 loss: 1.5337 
2024-11-08 13:27:11,289 - INFO - Epoch: 1/5, Iter: 13/93 -- label: 3.0000 loss: 1.4966 
2024-11-08 13:27:11,459 - INFO - Epoch:

## View training in tensorboard

Please uncomment the following cell to load tensorboard results.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ./runs

Launching TensorBoard...

## Inference

First thing to do is to prepare the test dataset:

In [ ]:
dataset_dir = Path(root_dir, "MedNIST")
class_names = sorted(f"{x.name}" for x in dataset_dir.iterdir() if x.is_dir())
testdata = MedNISTDataset(root_dir=root_dir, transform=transform, section="test", download=False, runtime_cache=True)

Next, we're going to establish a `SupervisedEvaluator`. This evaluator will process all the files in the specified directory and persist the results into a CSV file. Validation handlers (val_handlers) will be utilized to load the checkpoint file, providing an error if any file is unavailable, and they will also save the classification outcomes.

In [10]:
evaluator = SupervisedEvaluator(
    device=torch.device("cuda:0"),
    val_data_loader=DataLoader(testdata, batch_size=1, num_workers=0),
    network=model,
    inferer=SimpleInferer(),
    postprocessing=AsDiscreted(keys="pred", argmax=True),
    val_handlers=[
        CheckpointLoader(load_path=f"{out_dir}/checkpoint.pt", load_dict={"model": model}),
        ClassificationSaver(
            batch_transform=lambda batch: batch[0]["image"].meta, output_transform=from_engine(["pred"])
        ),
    ],
)

evaluator.run()

INFO:ignite.engine.engine.SupervisedEvaluator:Engine run resuming from iteration 0, epoch 0 until 1 epochs
INFO:ignite.engine.engine.SupervisedEvaluator:Restored all variables from ./eval/checkpoint.pt
INFO:ignite.engine.engine.SupervisedEvaluator:Epoch[1] Complete. Time taken: 00:01:24.338
INFO:ignite.engine.engine.SupervisedEvaluator:Engine run complete. Time taken: 00:01:24.390


By default, the inference results are stored in a file named "predictions.csv". However, this output filename can be customized within the `ClassificationSaver` handler, according to your preferences.
Upon examining the output, one can note that the second column corresponds to the predicted class. A more discernable interpretation can be achieved by using these values as indices mapped to our predefined list of class names.

In [12]:
max_items_to_print = 10
for fn, idx in np.loadtxt("./predictions.csv", delimiter=",", dtype=str):
    print(fn, class_names[int(float(idx))])
    max_items_to_print -= 1
    if max_items_to_print == 0:
        break

/workspace/Data/MedNIST/AbdomenCT/006070.jpeg AbdomenCT
/workspace/Data/MedNIST/BreastMRI/006574.jpeg BreastMRI
/workspace/Data/MedNIST/ChestCT/009858.jpeg ChestCT
/workspace/Data/MedNIST/CXR/007398.jpeg CXR
/workspace/Data/MedNIST/Hand/005663.jpeg Hand
/workspace/Data/MedNIST/HeadCT/006896.jpeg HeadCT
/workspace/Data/MedNIST/HeadCT/007179.jpeg HeadCT
/workspace/Data/MedNIST/CXR/001190.jpeg CXR
/workspace/Data/MedNIST/ChestCT/005138.jpeg ChestCT
/workspace/Data/MedNIST/BreastMRI/000023.jpeg BreastMRI
